# Meta-Analysis

## Overview
Meta-analysis combines quantitative results across independent studies to obtain a pooled estimate of an effect with greater precision than any single study.

**Core components:**

| Component | Description |
|---|---|
| Effect size | Standardised measure (Cohen's d, Hedges' g, log odds ratio, log response ratio) |
| Within-study variance | Sampling error in each study's effect estimate |
| Between-study variance (τ²) | True heterogeneity in effects across studies |
| Fixed-effects model | Assumes one true effect; heterogeneity = sampling error only |
| Random-effects model | Assumes distribution of true effects; τ² estimated explicitly |

**In ecology:** the log response ratio (lnRR = log(treatment mean / control mean)) is the most common effect size for experimental ecology meta-analyses.

**Software:** `metafor` (frequentist, full-featured) and `brms` (Bayesian random-effects; fits as a hierarchical model). This notebook covers both.

**Reference:** Quinn & Keough (2002) ch. 19 (synthesis); Borenstein et al. (2009) for full treatment.

---

In [ ]:
library(tidyverse); library(metafor)
set.seed(88)
# 20 studies on effect of nutrient addition on plant biomass
# Effect size: log response ratio (lnRR = log(treat_mean / ctrl_mean))
n_studies <- 20

# Simulate heterogeneous effects (some studies positive, variance in true effects)
true_effects <- rnorm(n_studies, mean = 0.35, sd = 0.25)  # true heterogeneity

studies <- data.frame(
  study_id   = paste0("Study_", 1:n_studies),
  year       = sample(2005:2023, n_studies, replace = TRUE),
  ecosystem  = sample(c("grassland", "forest", "wetland"), n_studies, replace = TRUE),
  n_treat    = sample(c(5, 8, 10, 15, 20), n_studies, replace = TRUE),
  n_ctrl     = sample(c(5, 8, 10, 15, 20), n_studies, replace = TRUE),
  lnRR       = true_effects + rnorm(n_studies, 0, 0.15),  # add measurement error
  se_lnRR    = runif(n_studies, 0.08, 0.25)  # study-level sampling SE
)
studies$vi <- studies$se_lnRR^2  # variance = SE²

cat("Simulated meta-analysis dataset:\n")
print(head(studies[, c("study_id", "ecosystem", "lnRR", "se_lnRR")]))
cat("\nOverall lnRR range:", round(range(studies$lnRR), 3), "\n")

---
## Random-effects meta-analysis (metafor)

In [ ]:
# Random-effects model: REML estimator of τ²
ma_re <- rma(yi = lnRR, vi = vi, data = studies,
             method = "REML", slab = study_id)

cat("=== Random-effects meta-analysis ===\n")
print(summary(ma_re))

cat("\n--- Key statistics ---\n")
cat(sprintf("Pooled lnRR = %.3f [95%% CI: %.3f, %.3f]\n",
            ma_re$beta, ma_re$ci.lb, ma_re$ci.ub))
cat(sprintf("Back-transformed (response ratio) = %.2fx\n", exp(ma_re$beta)))
cat(sprintf("Between-study heterogeneity: τ² = %.4f, τ = %.4f\n",
            ma_re$tau2, sqrt(ma_re$tau2)))
cat(sprintf("I² = %.1f%% (proportion of variance due to heterogeneity)\n",
            ma_re$I2))
cat(sprintf("Q = %.2f (df=%d, p=%.4f) — test of heterogeneity\n",
            ma_re$QE, ma_re$k - 1, ma_re$QEp))

In [ ]:
# Forest plot
forest(ma_re,
       header = c("Study", "lnRR [95% CI]"),
       xlab = "Log Response Ratio (lnRR)",
       mlab = "Pooled estimate (RE model, REML)",
       col = "steelblue", border = "steelblue",
       colout = "grey30")
cat("Forest plot: each row = one study; size ∝ 1/variance (study weight)\n")
cat("Diamond = pooled estimate with 95% CI\n")

In [ ]:
# Funnel plot + Egger's test for publication bias
funnel(ma_re, main = "Funnel plot (check for publication bias)",
       xlab = "lnRR", ylab = "Standard Error")
abline(v = 0, lty = 2, col = "grey")

# Egger's test (regression of effect size on precision)
egg <- regtest(ma_re, model = "lm")
cat("\nEgger's test for funnel plot asymmetry:\n")
print(egg)
cat("(Significant = potential publication bias or small-study effects)\n")

---
## Moderator analysis (meta-regression)

In [ ]:
# Does effect size vary by ecosystem type? (categorical moderator)
ma_mod <- rma(yi = lnRR, vi = vi, mods = ~ ecosystem,
              data = studies, method = "REML")
cat("--- Meta-regression: ecosystem type as moderator ---\n")
print(summary(ma_mod))

# QM tests whether the moderator explains heterogeneity
cat(sprintf("\nQM = %.2f (df=%d, p=%.4f) — moderator significance\n",
            ma_mod$QM, ma_mod$m, ma_mod$QMp))
cat(sprintf("Residual I²  = %.1f%% (unexplained heterogeneity after moderator)\n",
            ma_mod$I2))

In [ ]:
# Bayesian random-effects meta-analysis via brms
# Equivalent to multilevel model: study_id as random effect
cat("--- Bayesian meta-analysis (brms) ---\n")
cat("Bayesian approach treats τ as a hyperparameter with a prior.\n")
cat("Particularly useful when k is small (few studies).\n\n")
cat("Example brms code (not executed — requires MCMC):\n")
cat("  library(brms)\n")
cat("  prior_tau <- prior(half_normal(0, 0.5), class = sd)\n")
cat("  ma_bayes <- brm(\n")
cat("    lnRR | se(se_lnRR) ~ 1 + (1 | study_id),\n")
cat("    data  = studies,\n")
cat("    prior = prior_tau,\n")
cat("    chains = 4, iter = 4000\n")
cat("  )\n")
cat("  # Pooled estimate: posterior of Intercept\n")
cat("  # Between-study SD: posterior of sd_study_id__Intercept\n")
cat("  # interpret as P(τ > 0 | data) for evidence of heterogeneity\n")

---
## Common Pitfalls

**1. Using fixed-effects when heterogeneity is present**
Fixed-effects meta-analysis assumes one true effect size across all studies. If studies differ in populations, methods, or durations (almost always in ecology), heterogeneity is real and random-effects is appropriate. Check I² and the Q test before choosing a model.

**2. Interpreting I² as a simple threshold**
Higgins' benchmarks (I² < 25% = low, 25–75% = moderate, > 75% = high) are heuristics, not rules. With very large total sample size, a small I² can represent substantial absolute heterogeneity (τ). Report both I² and the absolute τ estimate.

**3. Ignoring publication bias**
Studies with significant results are more likely to be published. Always produce a funnel plot and run Egger's test. If bias is present, consider trim-and-fill correction (`trimfill()`) or selection models.

**4. Computing effect sizes incorrectly**
Log response ratio (lnRR) requires positive means and is undefined when control mean = 0. Hedges' g requires SD; don't substitute SE. Use `escalc()` from `metafor` to compute effect sizes correctly with appropriate variance formulas.

**5. Using meta-analysis with very few studies**
With k < 5, τ² estimates are unstable and Egger's test has low power. Bayesian meta-analysis with an informative prior on τ is preferable in this situation.

**6. Conflating ecological meta-analysis lnRR with Cohen's d**
LnRR is a ratio measure (appropriate for experiments with a clear control); Cohen's d / Hedges' g are mean difference measures standardised by SD (appropriate when comparing groups without a natural control). They are not interchangeable and require different variance formulas.


---
*r_methods_library - Samantha McGarrigle*